# ICHAN-Secure: Logic-First Reproduction and Correction of BASMEDSecure

**Recommended paper title:** *ICHAN-Secure: An Integrity-Aware, Capacity-Bounded, Header-Framed, Adaptive, Native-Bit-Depth Framework for Medical Image Data Hiding*

This notebook contains two separate workflows:

1. **Workflow 1 — BASMEDSecure reproduction and audit:** reproduce the paper's intended logistic-regression and adaptive-LSB logic while documenting corrections needed to make the pseudocode executable.
2. **Workflow 2 — ICHAN-Secure logical correction:** remove redundant model training, frame the payload, add CRC32 validation, enforce capacity and dtype invariants, and support native DICOM bit depths.

> **Verification status:** all cells are intentionally unexecuted. Outputs and numerical claims must be generated during a separate debugging and experimentation session.

> **Security boundary:** Workflow 2 improves correctness, not cryptographic security. CRC32 detects accidental corruption but does not authenticate an attacker-controlled payload.

## Abstract

Medical-image data hiding must balance payload capacity with image fidelity, recoverability, statistical undetectability, and diagnostic preservation. This notebook reproduces and audits BASMEDSecure, an intensity-based adaptive least-significant-bit framework that uses logistic regression to classify pixels. The audit focuses on key-map alignment, payload length, capacity, partial chunks, model necessity, and native DICOM bit depth. It then develops ICHAN-Secure, a logic-first corrected baseline using explicit classification, self-describing payload frames, CRC32 validation, capacity checks, and DICOM-aware processing. Experiments are designed for matched images and payloads using extraction accuracy, bits per pixel, side-information cost, PSNR, SSIM, and visualization of embedding changes. No results are claimed until these cells are executed and debugged.

**Keywords:** medical image data hiding; DICOM; steganography; least significant bit; payload integrity; steganalysis.

## I. Introduction

### A. Background and Clinical Motivation

Medical-image exchange requires confidentiality, integrity, diagnostic preservation, provenance, and interoperability. Steganography conceals the existence of a payload; encryption conceals its meaning. Neither property automatically provides the other.

### B. Problem Statement

BASMEDSecure assigns one, two, or three LSBs according to fixed pixel-intensity groups. Its printed algorithms do not fully define key-map alignment, payload termination, final partial chunks, cover recovery, or DICOM bit-depth behavior. Its Python draft also initializes `key_array` as a dictionary and then calls `.append()`.

### C. Research Gap

Four properties must be evaluated separately: capacity, numerical image similarity, statistical detectability, and diagnostic preservation. PSNR and SSIM evaluate only part of image similarity; they do not establish confidentiality, integrity, or clinical safety.

### D. Research Questions

- **RQ1:** Is intended BASMEDSecure embedding and extraction logic internally consistent?
- **RQ2:** Does logistic regression add information beyond deterministic thresholds used to generate its labels?
- **RQ3:** Can payload framing and native-bit-depth handling improve correctness and reproducibility?
- **RQ4:** How do payload, image source, bit depth, and embedding case affect distortion and side-information cost?
- **RQ5:** Which security and clinical tests remain before deployment claims are defensible?

### E. Contributions

This notebook supplies a disclosed reproduction, a logical audit, a corrected baseline, dataset guidance, matched evaluation functions, and a roadmap toward reversible, authenticated, steganalysis-tested, and quantum-safe data hiding.

## II. Related Work and Critical Discussion

### A. LSB and Content-Adaptive Steganography

Classical LSB replacement offers high capacity but can disturb statistical relationships between bit planes. RS steganalysis directly demonstrates this weakness. UNIWARD provides a stronger comparison because it assigns embedding costs from directional residuals and avoids predictable regions rather than classifying sensitivity from intensity alone.

### B. Medical Image and DICOM Data Hiding

Relevant alternatives include integer-wavelet DICOM embedding, contour-based selection, and reversible patient-data hiding. Comparisons must record native DICOM use, payload framing, encryption, reversibility, and clinical evaluation rather than PSNR alone.

### C. Reversible Data Hiding

Difference expansion and histogram shifting can recover the original image exactly after extraction. This property directly addresses BASMEDSecure Algorithm II's unsupported claim that it outputs the cover image.

### D. Steganalysis and Security Evaluation

Visual similarity and security are different measurements. Later work must define an attacker, run steganalysis, measure detection error or ROC-AUC, test bit errors, and evaluate relevant transformations such as DICOM rewriting, compression, noise, and cropping.

### E. Quantum and Post-Quantum Directions

NEQR-based quantum LSB papers are directly related, but image preparation, gate depth, noise, and readout must be counted. ML-KEM is a separate post-quantum key-establishment direction that runs on classical computers. Neither direction is implemented in this logic-first notebook.

## III. Methodology

Two workflows use separate classes so corrected behavior cannot be confused with reproduction behavior. Shared helpers keep bit conversion, metric definitions, and result formats controlled across both workflows.

**Environment and shared definitions.**

Required packages: `numpy`, `matplotlib`, `scikit-learn`, and `scikit-image`. DICOM cells additionally require `pydicom>=3.0`. The notebook does not install packages automatically. Record exact package versions when experiments begin.

In [ ]:
from __future__ import annotations

import copy
import struct
import zlib
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Iterable

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm, ListedColormap
from sklearn.linear_model import LogisticRegression
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

try:
    import pydicom
    from pydicom.uid import generate_uid
except ImportError:
    pydicom = None
    generate_uid = None

The next cell defines shared data contracts and bit operations. Explicit contracts prevent three original ambiguities: whether payload length is measured in bytes or bits, whether depth-map position equals pixel position, and how a final chunk shorter than the nominal depth is handled.

`EmbeddingResult.depth_map` always has the same shape as the image. Each entry stores the **actual** number of bits written to that pixel, including a shorter final chunk.

In [ ]:
@dataclass(frozen=True)
class EmbeddingResult:
    stego: np.ndarray
    depth_map: np.ndarray
    case: int
    payload_bits: int
    capacity_bits: int
    modified_pixels: int
    metadata: dict[str, Any]


@dataclass(frozen=True)
class EvaluationResult:
    mse: float
    psnr_db: float
    ssim: float
    payload_bpp: float
    modified_fraction: float
    max_absolute_change: int
    depth_map_bits_packed: int
    depth_map_overhead_ratio: float


def validate_grayscale_integer(image: np.ndarray, *, name: str = "image") -> np.ndarray:
    array = np.asarray(image)
    if array.ndim != 2:
        raise ValueError(f"{name} must be a two-dimensional grayscale array")
    if array.dtype.kind not in "iu":
        raise TypeError(f"{name} must use a signed or unsigned integer dtype")
    if array.size == 0:
        raise ValueError(f"{name} must not be empty")
    return array


def bytes_to_bits(payload: bytes) -> np.ndarray:
    if not isinstance(payload, bytes):
        raise TypeError("payload must be bytes")
    if not payload:
        return np.empty(0, dtype=np.uint8)
    return np.unpackbits(np.frombuffer(payload, dtype=np.uint8))


def bits_to_bytes(bits: Iterable[int]) -> bytes:
    bit_array = np.asarray(list(bits), dtype=np.uint8)
    if bit_array.size % 8 != 0:
        raise ValueError("bit count must be divisible by eight")
    if bit_array.size and np.any(bit_array > 1):
        raise ValueError("bits must contain only zero and one")
    return np.packbits(bit_array).tobytes()


def bit_chunk_to_int(bits: np.ndarray) -> int:
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return value


def replace_lsb(pixel_value: int, chunk_bits: np.ndarray) -> int:
    depth = int(len(chunk_bits))
    if depth not in (1, 2, 3):
        raise ValueError("embedding depth must be one, two, or three bits")
    cleared = int(pixel_value) & ~((1 << depth) - 1)
    return cleared | bit_chunk_to_int(chunk_bits)


def extract_lsb(pixel_value: int, depth: int) -> list[int]:
    if depth not in (1, 2, 3):
        raise ValueError("extraction depth must be one, two, or three bits")
    value = int(pixel_value) & ((1 << depth) - 1)
    return [(value >> shift) & 1 for shift in range(depth - 1, -1, -1)]


def choose_embedding_case(bit_count: int, p0: int, p1: int, p2: int) -> tuple[int, int]:
    if bit_count < 0:
        raise ValueError("bit_count must not be negative")
    maximum = 3 * p0 + 2 * p1 + p2
    if bit_count > maximum:
        raise ValueError(f"payload requires {bit_count} bits but capacity is {maximum} bits")
    if bit_count <= p0:
        return 1, maximum
    if bit_count <= 2 * p0 + p1:
        return 2, maximum
    return 3, maximum


def depth_for_label(case: int, label: int) -> int:
    schedules = {
        1: {0: 1, 1: 0, 2: 0},
        2: {0: 2, 1: 1, 2: 0},
        3: {0: 3, 1: 2, 2: 1},
    }
    try:
        return schedules[case][int(label)]
    except KeyError as error:
        raise ValueError(f"unsupported case or label: case={case}, label={label}") from error

### A. Workflow 1 — BASMEDSecure Reproduction

#### Analytical step

The paper creates labels from fixed 8-bit thresholds and trains logistic regression on the same scalar pixel values. This implementation retains that behavior so model predictions can later be compared against the source labels. It deliberately fixes only execution-blocking ambiguity:

- depth map is a full `uint8` array, not a dictionary;
- all skipped pixels retain depth zero;
- cases are mutually exclusive;
- capacity is checked before mutation;
- final chunks use their actual remaining depth;
- extraction receives explicit payload bit length because the paper embeds no header.

These are reproduction corrections, not evidence that printed pseudocode was correct.

In [ ]:
class PaperFaithfulBASMEDSecure:
    """Executable interpretation of the paper's intended 8-bit method."""

    def __init__(self, *, max_iter: int = 1500) -> None:
        self.model = LogisticRegression(max_iter=max_iter, solver="lbfgs")
        self.is_trained = False

    @staticmethod
    def deterministic_labels(pixel_values: np.ndarray) -> np.ndarray:
        values = np.asarray(pixel_values)
        labels = np.zeros(values.shape, dtype=np.uint8)
        labels[(values >= 101) & (values <= 150)] = 1
        labels[values >= 151] = 2
        return labels

    def fit(self, cover: np.ndarray) -> None:
        image = validate_grayscale_integer(cover, name="cover")
        if image.dtype != np.uint8:
            raise TypeError("paper-faithful workflow is limited to uint8 images")
        pixels = image.reshape(-1, 1)
        labels = self.deterministic_labels(pixels).ravel()
        present_classes = np.unique(labels)
        if present_classes.size < 2:
            raise ValueError(
                "logistic regression requires at least two classes; "
                f"this image contains classes {present_classes.tolist()}"
            )
        self.model.fit(pixels, labels)
        self.is_trained = True

    def predict_labels(self, cover: np.ndarray) -> np.ndarray:
        image = validate_grayscale_integer(cover, name="cover")
        if not self.is_trained:
            raise RuntimeError("fit must be called before prediction")
        return self.model.predict(image.reshape(-1, 1)).astype(np.uint8)

    def embed(self, cover: np.ndarray, payload: bytes) -> EmbeddingResult:
        image = validate_grayscale_integer(cover, name="cover")
        if image.dtype != np.uint8:
            raise TypeError("paper-faithful workflow is limited to uint8 images")
        self.fit(image)
        labels = self.predict_labels(image)
        counts = np.bincount(labels, minlength=3)
        p0, p1, p2 = (int(counts[0]), int(counts[1]), int(counts[2]))
        secret_bits = bytes_to_bits(payload)
        case, capacity = choose_embedding_case(len(secret_bits), p0, p1, p2)

        original_flat = image.ravel()
        stego_flat = original_flat.copy()
        depth_flat = np.zeros(stego_flat.size, dtype=np.uint8)
        bit_index = 0

        for pixel_index, label in enumerate(labels):
            if bit_index >= len(secret_bits):
                break
            nominal_depth = depth_for_label(case, int(label))
            if nominal_depth == 0:
                continue
            actual_depth = min(nominal_depth, len(secret_bits) - bit_index)
            chunk = secret_bits[bit_index : bit_index + actual_depth]
            stego_flat[pixel_index] = replace_lsb(int(stego_flat[pixel_index]), chunk)
            depth_flat[pixel_index] = actual_depth
            bit_index += actual_depth

        if bit_index != len(secret_bits):
            raise RuntimeError("internal embedding invariant failed: not all bits were consumed")

        stego = stego_flat.reshape(image.shape)
        source_labels = self.deterministic_labels(image.ravel()).ravel()
        agreement = float(np.mean(labels == source_labels))
        return EmbeddingResult(
            stego=stego,
            depth_map=depth_flat.reshape(image.shape),
            case=case,
            payload_bits=len(secret_bits),
            capacity_bits=capacity,
            modified_pixels=int(np.count_nonzero(stego_flat != original_flat)),
            metadata={
                "predicted_class_counts": (p0, p1, p2),
                "classifier_threshold_agreement": agreement,
                "workflow": "paper-faithful",
            },
        )

    @staticmethod
    def extract(stego: np.ndarray, depth_map: np.ndarray, payload_bit_length: int) -> bytes:
        image = validate_grayscale_integer(stego, name="stego")
        depths = np.asarray(depth_map)
        if depths.shape != image.shape:
            raise ValueError("depth_map shape must match stego image shape")
        if payload_bit_length < 0 or payload_bit_length % 8 != 0:
            raise ValueError("payload_bit_length must be a non-negative multiple of eight")

        recovered_bits: list[int] = []
        for pixel_value, depth in zip(image.ravel(), depths.ravel()):
            if len(recovered_bits) >= payload_bit_length:
                break
            if int(depth) > 0:
                recovered_bits.extend(extract_lsb(int(pixel_value), int(depth)))

        if len(recovered_bits) < payload_bit_length:
            raise ValueError("depth map contains fewer bits than requested payload length")
        return bits_to_bytes(recovered_bits[:payload_bit_length])

#### Dataset source for Workflow 1

Start with a synthetic 8-bit array only to debug algorithm flow. Then use an explicitly documented 8-bit conversion of a DICOM image, retaining the original native array separately. Search for **`pydicom examples get_path ct`** for a small DICOM input. Do not report this example as a dataset or generalization experiment.

The next cell is a runnable example for your later debugging session. It is not executed here.

In [ ]:
paper_rng = np.random.default_rng(99)
paper_cover = paper_rng.integers(0, 256, size=(512, 512), dtype=np.uint8)
paper_payload = (
    b"RESEARCH-ONLY SYNTHETIC MEDICAL RECORD | "
    b"NO REAL PATIENT INFORMATION | BASMEDSecure reproduction"
)

paper_method = PaperFaithfulBASMEDSecure()
paper_result = paper_method.embed(paper_cover, paper_payload)
paper_recovered = paper_method.extract(
    paper_result.stego,
    paper_result.depth_map,
    paper_result.payload_bits,
)

paper_example_summary = {
    "case": paper_result.case,
    "payload_bits": paper_result.payload_bits,
    "capacity_bits": paper_result.capacity_bits,
    "modified_pixels": paper_result.modified_pixels,
    "exact_recovery": paper_recovered == paper_payload,
    **paper_result.metadata,
}
paper_example_summary

#### Evaluation logic

Metrics must use the declared sample range, not a fixed value of 255. For `BitsStored=b`, unsigned and signed integer domains both span `2**b - 1` possible intensity steps. Side-map overhead is reported using the theoretical two-bit packing required for values 0–3; Python object or `uint8` storage may be larger.

In [ ]:
def integer_domain(bits_stored: int, signed: bool) -> tuple[int, int]:
    if isinstance(bits_stored, bool) or not isinstance(bits_stored, int):
        raise TypeError("bits_stored must be an integer")
    if bits_stored < 1 or bits_stored > 32:
        raise ValueError("bits_stored must be from 1 through 32")
    if signed:
        return -(1 << (bits_stored - 1)), (1 << (bits_stored - 1)) - 1
    return 0, (1 << bits_stored) - 1


def bit_error_rate(expected: bytes, recovered: bytes) -> float:
    expected_bits = bytes_to_bits(expected)
    recovered_bits = bytes_to_bits(recovered)
    total = max(len(expected_bits), len(recovered_bits))
    if total == 0:
        return 0.0
    shared = min(len(expected_bits), len(recovered_bits))
    mismatches = int(np.count_nonzero(expected_bits[:shared] != recovered_bits[:shared]))
    mismatches += total - shared
    return mismatches / total


def evaluate_embedding(
    cover: np.ndarray,
    result: EmbeddingResult,
    *,
    bits_stored: int,
    signed: bool,
) -> EvaluationResult:
    original = validate_grayscale_integer(cover, name="cover")
    stego = validate_grayscale_integer(result.stego, name="stego")
    if original.shape != stego.shape:
        raise ValueError("cover and stego shapes must match")
    _, domain_max = integer_domain(bits_stored, signed)
    domain_min, _ = integer_domain(bits_stored, signed)
    data_range = domain_max - domain_min

    difference = stego.astype(np.int64) - original.astype(np.int64)
    mse = float(np.mean(difference.astype(np.float64) ** 2))
    min_side = min(original.shape)
    win_size = None
    if min_side < 7:
        win_size = min_side if min_side % 2 == 1 else min_side - 1
        if win_size < 3:
            raise ValueError("SSIM requires each image dimension to be at least three")

    psnr_value = float(peak_signal_noise_ratio(original, stego, data_range=data_range))
    ssim_value = float(
        structural_similarity(original, stego, data_range=data_range, win_size=win_size)
    )
    packed_map_bits = int(result.depth_map.size * 2)
    overhead = float("inf") if result.payload_bits == 0 else packed_map_bits / result.payload_bits
    return EvaluationResult(
        mse=mse,
        psnr_db=psnr_value,
        ssim=ssim_value,
        payload_bpp=result.payload_bits / original.size,
        modified_fraction=result.modified_pixels / original.size,
        max_absolute_change=int(np.max(np.abs(difference))),
        depth_map_bits_packed=packed_map_bits,
        depth_map_overhead_ratio=overhead,
    )

#### Visualization logic

Different datasets can use different stored ranges. Cover and stego images must share the same grayscale limits; otherwise automatic plotting can exaggerate or hide changes. Difference maps need a separate scale. Label and depth maps are categorical, so they use discrete colors. Histograms use native values and at most 256 bins for readability.

In [ ]:
def plot_cover_stego_difference(
    cover: np.ndarray,
    stego: np.ndarray,
    *,
    bits_stored: int,
    signed: bool,
    title: str = "Embedding inspection",
) -> tuple[plt.Figure, np.ndarray]:
    original = validate_grayscale_integer(cover, name="cover")
    modified = validate_grayscale_integer(stego, name="stego")
    if original.shape != modified.shape:
        raise ValueError("cover and stego shapes must match")
    domain_min, domain_max = integer_domain(bits_stored, signed)
    difference = np.abs(modified.astype(np.int64) - original.astype(np.int64))

    figure, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    axes[0].imshow(original, cmap="gray", vmin=domain_min, vmax=domain_max)
    axes[0].set_title("Cover")
    axes[1].imshow(modified, cmap="gray", vmin=domain_min, vmax=domain_max)
    axes[1].set_title("Stego")
    difference_image = axes[2].imshow(difference, cmap="inferno", vmin=0)
    axes[2].set_title("Absolute difference")
    figure.colorbar(difference_image, ax=axes[2], fraction=0.046, pad=0.04)
    for axis in axes:
        axis.axis("off")
    figure.suptitle(title)
    return figure, axes


def plot_label_and_depth_maps(
    labels: np.ndarray, depth_map: np.ndarray
) -> tuple[plt.Figure, np.ndarray]:
    label_array = np.asarray(labels)
    depth_array = np.asarray(depth_map)
    if label_array.shape != depth_array.shape:
        raise ValueError("labels and depth_map shapes must match")

    label_cmap = ListedColormap(["#264653", "#e9c46a", "#e76f51"])
    depth_cmap = ListedColormap(["#f7f7f7", "#91bfdb", "#fc8d59", "#d73027"])
    figure, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    label_image = axes[0].imshow(
        label_array, cmap=label_cmap, norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5], 3)
    )
    depth_image = axes[1].imshow(
        depth_array, cmap=depth_cmap, norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], 4)
    )
    axes[0].set_title("Intensity label: 0, 1, 2")
    axes[1].set_title("Embedded bits per pixel: 0–3")
    figure.colorbar(label_image, ax=axes[0], ticks=[0, 1, 2], fraction=0.046, pad=0.04)
    figure.colorbar(depth_image, ax=axes[1], ticks=[0, 1, 2, 3], fraction=0.046, pad=0.04)
    for axis in axes:
        axis.axis("off")
    return figure, axes


def plot_native_histograms(
    cover: np.ndarray,
    stego: np.ndarray,
    *,
    bits_stored: int,
    signed: bool,
) -> tuple[plt.Figure, plt.Axes]:
    original = validate_grayscale_integer(cover, name="cover")
    modified = validate_grayscale_integer(stego, name="stego")
    domain_min, domain_max = integer_domain(bits_stored, signed)
    bins = min(256, 1 << min(bits_stored, 16))
    figure, axis = plt.subplots(figsize=(10, 4), constrained_layout=True)
    axis.hist(
        original.ravel(), bins=bins, range=(domain_min, domain_max),
        histtype="step", linewidth=1.5, label="Cover"
    )
    axis.hist(
        modified.ravel(), bins=bins, range=(domain_min, domain_max),
        histtype="step", linewidth=1.5, label="Stego"
    )
    axis.set_xlabel("Stored pixel value")
    axis.set_ylabel("Pixel count")
    axis.set_title("Native-domain intensity histogram")
    axis.legend()
    return figure, axis

### B. Logical Audit of BASMEDSecure

Record these findings beside Workflow 1 results:

1. Original Python code cannot append to `key_array = {}`.
2. Printed Algorithm I does not append zero for skipped positions, conflicting with an image-shaped key map.
3. Algorithm II needs secret length but does not receive or recover it.
4. Algorithm II cannot reconstruct the cover because original LSB values are not retained.
5. Logistic regression learns labels generated from fixed thresholds and is not used during extraction.
6. Fixed 8-bit ranges do not represent general DICOM pixel domains.
7. A full two-bit-per-pixel depth map may exceed the payload itself.
8. Sequential LSB replacement remains statistically detectable and unencrypted.

The next cell converts these findings into machine-readable audit records for a later results table; it does not test the implementation.

In [ ]:
BASMEDSECURE_AUDIT = [
    {"severity": "Critical", "area": "Reference code", "finding": "dictionary used as appendable key array", "response": "repair only in disclosed reproduction"},
    {"severity": "Critical", "area": "Reversibility", "finding": "cover recovery claimed without original LSB information", "response": "withdraw claim or use genuine RDH"},
    {"severity": "High", "area": "Extraction", "finding": "payload length unavailable to Algorithm II", "response": "add a self-describing frame"},
    {"severity": "High", "area": "Key map", "finding": "skipped pixel positions are not represented consistently", "response": "use image-aligned map or keyed traversal"},
    {"severity": "High", "area": "Machine learning", "finding": "model relearns deterministic thresholds", "response": "use direct rules unless external labels justify ML"},
    {"severity": "High", "area": "DICOM", "finding": "fixed uint8 assumptions ignore native bit depth and signed pixels", "response": "preserve declared native integer domain"},
    {"severity": "High", "area": "Security", "finding": "no encryption, authentication, secret placement key, threat model, or steganalysis", "response": "add these before security claims"},
    {"severity": "High", "area": "Dataset", "finding": "no patient-level split or independent external validation", "response": "split and aggregate by patient"},
    {"severity": "High", "area": "Evaluation", "finding": "PSNR and SSIM cannot establish undetectability or diagnostic safety", "response": "add steganalysis and task-based evaluation"},
    {"severity": "Moderate", "area": "Efficiency", "finding": "side-information map may exceed payload size", "response": "report and later reduce map overhead"},
    {"severity": "Moderate", "area": "Metrics", "finding": "fixed or unclear data ranges invalidate cross-bit-depth comparisons", "response": "use declared native data range"},
    {"severity": "Low", "area": "Reproducibility", "finding": "seeds, versions, and failure behavior are incomplete", "response": "publish environment and experiment manifest"},
]
BASMEDSECURE_AUDIT

#### Risk and improvement grades

Risk grades mean: **Critical** contradicts a core correctness claim or prevents execution; **High** materially invalidates a security, generality, or evaluation claim; **Moderate** creates important overhead or bias; **Low** mainly affects reporting or maintainability.

Improvement levels mean: **I1** reporting/usability, **I2** reproducibility/measurement, **I3** core correctness or input-generalization repair, and **I4** a trust or clinical-safety change requiring a new validation campaign. Workflow 2 is graded **I3 overall for logic**, but only **I1–I2 for security** because it does not yet add authentication, secrecy, reversible embedding, steganalysis, or clinical validation.

In [ ]:
ICHAN_IMPROVEMENTS = [
    {"change": "direct native-domain classification", "level": "I2", "residual_risk": "unvalidated intensity heuristic"},
    {"change": "length/version/magic frame", "level": "I3", "residual_risk": "visible and unauthenticated header"},
    {"change": "exact depth-map alignment and partial chunks", "level": "I3", "residual_risk": "external map remains costly and sensitive"},
    {"change": "pre-mutation capacity and dtype validation", "level": "I3", "residual_risk": "embedding remains irreversible"},
    {"change": "native bit depth and conservative DICOM handling", "level": "I3", "residual_risk": "not all IODs or transfer syntaxes supported"},
    {"change": "CRC32 corruption detection", "level": "I2", "residual_risk": "not cryptographic authentication"},
    {"change": "patient-level external-validation protocol", "level": "I2", "residual_risk": "experiments and clinical validation remain pending"},
]
ICHAN_IMPROVEMENTS

### C. Proposed ICHAN-Secure Workflow

This pass formally specifies **Step 1** and **Step 2**. Step 1 maps every valid stored pixel to one deterministic class. Step 2 converts arbitrary payload bytes into a self-describing frame. The later `ICHANSecure` class consumes these interfaces, but placement, reversible cover recovery, and steganographic security are outside the present proof boundary.

### D. Step 1 — Deterministic Native-Domain Classification

**Objective.** Produce exactly one label for every valid pixel without fitting a classifier whose target was generated from the same pixel intensity. The proof establishes reproduction of the deterministic labeling rule only; it does not establish clinical relevance.

**Notation and domain.** Let the image be $X=[x_{i,j}]\in\mathcal{D}_{b,s}^{H\times W}$ with $N=HW$ pixels, stored bit depth $b$, and signedness $s\in\{0,1\}$. The stored-value domain is

$$\mathcal{D}_{b,s}=\{d_{\min},\ldots,d_{\max}\},\qquad(d_{\min},d_{\max})=\begin{cases}(0,2^b-1),&s=0,\\(-2^{b-1},2^{b-1}-1),&s=1.\end{cases}\tag{1}$$

Both cases have span $R=d_{\max}-d_{\min}=2^b-1$. Scale the BASMEDSecure cutoffs into this domain:

$$t_0=d_{\min}+\left\lfloor\frac{100R}{255}\right\rfloor,\qquad t_1=d_{\min}+\left\lfloor\frac{150R}{255}\right\rfloor.\tag{2}$$

Define $g:\mathcal{D}_{b,s}\rightarrow\{0,1,2\}$ by

$$g(x)=\begin{cases}0,&d_{\min}\leq x\leq t_0,\\1,&t_0<x\leq t_1,\\2,&t_1<x\leq d_{\max}.\end{cases}\tag{3}$$

The label map is $Y=[g(x_{i,j})]$. Its class counts are $P_k=\sum_i\sum_j\mathbf{1}[g(x_{i,j})=k]$ for $k\in\{0,1,2\}$, so $P_0+P_1+P_2=N$.

**Algorithm 1 — Native-domain classification.**

```text
Require: non-empty integer image X, stored bit depth b, signedness s
Ensure: label map Y and thresholds (t0, t1)
1: Validate dimensionality, integer dtype, b, signedness, and observed range.
2: Derive (dmin, dmax) from (b, s); set R <- dmax - dmin.
3: Set t0 <- dmin + floor(100R/255).
4: Set t1 <- dmin + floor(150R/255).
5: For each x(i,j):
6:     if x(i,j) <= t0, set Y(i,j) <- 0.
7:     else if x(i,j) <= t1, set Y(i,j) <- 1.
8:     else set Y(i,j) <- 2.
9: Return Y and (t0, t1).
```

**Proposition 1 — Totality and uniqueness.** Every valid pixel receives exactly one label. **Proof.** Since $t_0\leq t_1$, the three predicates in (3) are pairwise disjoint and their union is the entire declared domain. If a very small bit depth makes $t_0=t_1$, class 1 is empty, but the remaining branches are still disjoint and exhaustive. $\square$

**Proposition 2 — Exact unsigned 8-bit reduction.** For $b=8$ and $s=0$, $R=255$, giving $t_0=\lfloor100(255)/255\rfloor=100$ and $t_1=\lfloor150(255)/255\rfloor=150$. Thus (3) exactly reproduces the original threshold intervals. $\square$

**Proposition 3 — Order preservation.** If $x_a\leq x_b$, then $g(x_a)\leq g(x_b)$. **Proof.** Increasing a stored value can remain in one ordered interval or cross a threshold toward a higher label; it cannot cross a threshold in reverse. $\square$

**Proposition 4 — Deterministic-label consistency.** When target labels are defined by $y_n=g(x_n)$, direct classification has empirical disagreement

$$\widehat{R}(g)=\frac{1}{N}\sum_{n=1}^{N}\mathbf{1}[g(x_n)\neq y_n]=0.\tag{4}$$

**Proof.** Substitution of $y_n=g(x_n)$ makes every indicator zero. A fitted logistic-regression model cannot improve disagreement below zero; it can only approximate or depart from the predefined rule. This proves training is unnecessary for reproducing those generated targets, not that the rule is clinically valid. $\square$

**Complexity.** Threshold calculation is $O(1)$. For $N$ pixels, classification takes $\Theta(N)$ time and $\Theta(N)$ output space for $Y$, with $O(1)$ auxiliary state when output storage is excluded.

**Dataset and measurement boundary.** Synthetic boundary values are sufficient for these proofs. For distribution studies, search **`TCIA LIDC-IDRI DICOM download`** and retain `BitsStored`, `BitsAllocated`, `PixelRepresentation`, `RescaleSlope`, and `RescaleIntercept`. Equation (3) operates on stored values; it must not be described as a Hounsfield-unit or tissue-sensitivity classifier.

In [ ]:
BASMED_LOW_NUMERATOR = 100
BASMED_MEDIUM_NUMERATOR = 150
BASMED_REFERENCE_MAX = 255


def scaled_basmed_cutoffs(bits_stored: int, signed: bool) -> tuple[int, int]:
    domain_min, domain_max = integer_domain(bits_stored, signed)
    span = domain_max - domain_min
    low_cutoff = domain_min + (BASMED_LOW_NUMERATOR * span) // BASMED_REFERENCE_MAX
    medium_cutoff = domain_min + (BASMED_MEDIUM_NUMERATOR * span) // BASMED_REFERENCE_MAX
    return low_cutoff, medium_cutoff


def classify_native_intensity(
    image: np.ndarray, *, bits_stored: int, signed: bool
) -> tuple[np.ndarray, tuple[int, int]]:
    array = validate_grayscale_integer(image, name="image")
    if not isinstance(signed, (bool, np.bool_)):
        raise TypeError("signed must be a boolean")
    if bits_stored > array.dtype.itemsize * 8:
        raise ValueError("bits_stored exceeds image dtype width")
    expected_kind = "i" if bool(signed) else "u"
    if array.dtype.kind != expected_kind:
        expected_name = "signed" if bool(signed) else "unsigned"
        raise TypeError(f"declared {expected_name} pixels require a matching NumPy dtype")
    domain_min, domain_max = integer_domain(bits_stored, signed)
    observed_min = int(array.min())
    observed_max = int(array.max())
    if observed_min < domain_min or observed_max > domain_max:
        raise ValueError(
            f"pixel values [{observed_min}, {observed_max}] exceed declared "
            f"domain [{domain_min}, {domain_max}]"
        )
    low_cutoff, medium_cutoff = scaled_basmed_cutoffs(bits_stored, bool(signed))
    labels = np.zeros(array.shape, dtype=np.uint8)
    labels[(array > low_cutoff) & (array <= medium_cutoff)] = 1
    labels[array > medium_cutoff] = 2
    return labels, (low_cutoff, medium_cutoff)


def summarize_native_classification(
    image: np.ndarray, *, bits_stored: int, signed: bool
) -> dict[str, Any]:
    labels, cutoffs = classify_native_intensity(
        image, bits_stored=bits_stored, signed=signed
    )
    counts = np.bincount(labels.ravel(), minlength=3)
    total = int(labels.size)
    return {
        "bits_stored": bits_stored,
        "signed": bool(signed),
        "cutoffs": cutoffs,
        "counts": tuple(int(value) for value in counts),
        "fractions": tuple(float(value / total) for value in counts),
    }

**Worked derivation and proof mapping.** For unsigned 12-bit data, $d_{\min}=0$, $d_{\max}=4095$, and $R=4095$. Therefore, $t_0=\lfloor409500/255\rfloor=1605$ and $t_1=\lfloor614250/255\rfloor=2408$. For signed 12-bit data, translating the same span to $[-2048,2047]$ gives $t_0=-443$ and $t_1=360$.

The next unexecuted assertions map implementation evidence to the proof obligations:

| Assertion | Mathematical obligation |
|---|---|
| shape preservation and labels in `{0,1,2}` | Proposition 1 |
| cutoffs `(100,150)` and exact boundary labels | Proposition 2 |
| nondecreasing labels for ordered boundary inputs | Proposition 3 |
| direct labels equal their defining targets | Proposition 4 |
| 12-bit cutoffs `(1605,2408)` and `(-443,360)` | native-domain derivation |

Passing these assertions would verify the implementation against the specification for the selected boundaries. It would not validate performance on a clinical population.

In [ ]:
step1_uint8_values = np.array([[0, 100, 101, 150, 151, 255]], dtype=np.uint8)
step1_uint8_labels, step1_uint8_cutoffs = classify_native_intensity(
    step1_uint8_values, bits_stored=8, signed=False
)
step1_expected_labels = np.array([[0, 0, 1, 1, 2, 2]], dtype=np.uint8)

step1_12bit_values = np.array(
    [[0, 1605, 1606, 2408, 2409, 4095]], dtype=np.uint16
)
step1_12bit_labels, step1_12bit_cutoffs = classify_native_intensity(
    step1_12bit_values, bits_stored=12, signed=False
)

step1_signed_12bit_values = np.array(
    [[-2048, -443, -442, 360, 361, 2047]], dtype=np.int16
)
step1_signed_labels, step1_signed_cutoffs = classify_native_intensity(
    step1_signed_12bit_values, bits_stored=12, signed=True
)

assert step1_uint8_cutoffs == (100, 150)
assert np.array_equal(step1_uint8_labels, step1_expected_labels)
assert step1_12bit_cutoffs == (1605, 2408)
assert step1_signed_cutoffs == (-443, 360)
assert step1_uint8_labels.shape == step1_uint8_values.shape
assert set(np.unique(step1_uint8_labels)).issubset({0, 1, 2})
assert np.all(np.diff(step1_uint8_labels.astype(np.int16).ravel()) >= 0)
assert np.array_equal(
    step1_signed_labels, np.array([[0, 0, 1, 1, 2, 2]], dtype=np.uint8)
)
assert np.array_equal(step1_uint8_labels, classify_native_intensity(
    step1_uint8_values, bits_stored=8, signed=False
)[0])

step1_boundary_summary = {
    "uint8_cutoffs": step1_uint8_cutoffs,
    "uint8_labels": step1_uint8_labels.tolist(),
    "uint12_cutoffs": step1_12bit_cutoffs,
    "uint12_labels": step1_12bit_labels.tolist(),
    "signed12_cutoffs": step1_signed_cutoffs,
    "signed12_labels": step1_signed_labels.tolist(),
}
step1_boundary_summary

**Mathematical visualization — proposed Fig. 1.** The following number-line diagram displays the complete declared domain, the two derived thresholds, and the three decision sets from (3). Text labels and hatch patterns duplicate the color encoding so the boundary logic remains interpretable in grayscale. Proposed caption: **Native-domain decision regions used by Step 1, including scaled thresholds and boundary inclusivity.** The figure is dataset-independent and may be exported later at the resolution and file type required by the selected IEEE venue.

In [ ]:
def plot_step1_decision_regions(
    *, bits_stored: int, signed: bool
) -> tuple[plt.Figure, plt.Axes]:
    domain_min, domain_max = integer_domain(bits_stored, signed)
    low_cutoff, medium_cutoff = scaled_basmed_cutoffs(bits_stored, signed)
    regions = [
        (domain_min, low_cutoff, "Class 0: x <= t0", "#2b83ba", "///"),
        (low_cutoff, medium_cutoff, "Class 1: t0 < x <= t1", "#fdae61", "..."),
        (medium_cutoff, domain_max, "Class 2: x > t1", "#d7191c", "xxx"),
    ]

    figure, axis = plt.subplots(figsize=(13, 3), constrained_layout=True)
    for left, right, label, color, hatch in regions:
        width = max(right - left, 0)
        axis.barh(
            y=0, width=width, left=left, height=0.42,
            color=color, edgecolor="black", alpha=0.65, hatch=hatch, label=label
        )
    axis.axvline(low_cutoff, color="black", linestyle="--", linewidth=1.2)
    axis.axvline(medium_cutoff, color="black", linestyle=":", linewidth=1.5)
    axis.annotate(
        f"t0={low_cutoff} (included in class 0)",
        xy=(low_cutoff, 0.22), xytext=(low_cutoff, 0.55),
        ha="center", arrowprops={"arrowstyle": "->"},
    )
    axis.annotate(
        f"t1={medium_cutoff} (included in class 1)",
        xy=(medium_cutoff, 0.22), xytext=(medium_cutoff, -0.55),
        ha="center", arrowprops={"arrowstyle": "->"},
    )
    axis.set(
        xlim=(domain_min, domain_max), ylim=(-0.8, 0.8), yticks=[],
        xlabel="stored pixel value x",
        title=f"Step 1 decision regions: b={bits_stored}, signed={signed}",
    )
    axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3)
    return figure, axis


step1_math_figure, step1_math_axis = plot_step1_decision_regions(
    bits_stored=12, signed=False
)

**Visualization choice.** Use the same three-panel diagnostic across datasets: raw stored values, a discrete class map, and a histogram with both cutoffs. The function derives its display and histogram range from the declared native domain, so 8-, 12-, and 16-bit inputs are not silently normalized to 0–255. For multi-dataset results, compare class fractions at patient level rather than treating correlated slices as independent samples.

In [ ]:
def plot_native_classification(
    image: np.ndarray,
    *,
    bits_stored: int,
    signed: bool,
    title: str = "ICHAN-Secure Step 1",
) -> tuple[plt.Figure, np.ndarray]:
    array = validate_grayscale_integer(image, name="image")
    labels, (low_cutoff, medium_cutoff) = classify_native_intensity(
        array, bits_stored=bits_stored, signed=signed
    )
    domain_min, domain_max = integer_domain(bits_stored, signed)
    label_cmap = ListedColormap(["#2b83ba", "#fdae61", "#d7191c"])
    label_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], label_cmap.N)
    histogram_bins = int(np.clip(np.sqrt(array.size), 32, 256))

    figure, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
    raw_image = axes[0].imshow(
        array, cmap="gray", vmin=domain_min, vmax=domain_max
    )
    axes[0].set_title("Stored pixel values")
    axes[0].axis("off")
    figure.colorbar(raw_image, ax=axes[0], shrink=0.8, label="stored value")

    label_image = axes[1].imshow(labels, cmap=label_cmap, norm=label_norm)
    axes[1].set_title("Deterministic class map")
    axes[1].axis("off")
    label_bar = figure.colorbar(label_image, ax=axes[1], ticks=[0, 1, 2], shrink=0.8)
    label_bar.ax.set_yticklabels(["0: low", "1: medium", "2: high"])

    axes[2].hist(
        array.ravel(),
        bins=histogram_bins,
        range=(domain_min, domain_max),
        histtype="step",
        linewidth=1.5,
        color="black",
    )
    axes[2].axvline(low_cutoff, color="#2b83ba", linestyle="--", label=f"t0={low_cutoff}")
    axes[2].axvline(medium_cutoff, color="#d7191c", linestyle="--", label=f"t1={medium_cutoff}")
    axes[2].set(
        title="Native-domain histogram",
        xlabel="stored pixel value",
        ylabel="pixel count",
        xlim=(domain_min, domain_max),
    )
    axes[2].legend()
    figure.suptitle(title)
    return figure, axes

In [ ]:
# Dataset search after this controlled check: TCIA LIDC-IDRI DICOM download
step1_gradient = np.linspace(0, 4095, num=256, dtype=np.uint16)
step1_demo_image = np.tile(step1_gradient, (128, 1))
step1_demo_summary = summarize_native_classification(
    step1_demo_image, bits_stored=12, signed=False
)
step1_figure, step1_axes = plot_native_classification(
    step1_demo_image,
    bits_stored=12,
    signed=False,
    title="Step 1 controlled 12-bit classification",
)
step1_demo_summary

### E. Step 2 — Self-Describing Payload Framing

**Objective.** Remove Algorithm II's hidden dependency on an externally supplied payload length. This step defines a portable byte protocol before the frame is converted to embedding bits.

**Notation and frame definition.** Let $P=(p_0,\ldots,p_{L-1})\in\{0,\ldots,255\}^L$ be a payload of $L$ bytes, where $0\leq L\leq2^{64}-1$. Let $M=\texttt{ICHS}$, $V=1$, $\operatorname{len}_{64}(L)$ be the eight-byte unsigned big-endian representation of $L$, and $C(P)$ be its four-byte CRC32 value. With $\Vert$ denoting byte concatenation,

$$F(P)=M\Vert V\Vert\operatorname{len}_{64}(L)\Vert C(P)\Vert P.\tag{5}$$

The header has $4+1+8+4=17$ bytes; consequently,

$$|F(P)|=L+17\text{ bytes}=8L+136\text{ bits}.\tag{6}$$

| Field | Bytes | Offsets | Acceptance predicate |
|---|---:|---:|---|
| Magic `ICHS` | 4 | 0–3 | exact match |
| Version | 1 | 4 | supported version 1 |
| Payload length | 8 | 5–12 | received body length equals $L$ |
| Payload CRC32 | 4 | 13–16 | recomputed value equals header value |
| Payload | $L$ | 17–$16+L$ | returned only after all predicates pass |

**Algorithm 2A — Frame construction.**

```text
Require: byte string P
Ensure: byte frame F
1: Reject P unless it is bytes and |P| <= 2^64 - 1.
2: L <- |P|; c <- CRC32(P).
3: H <- ICHS || 0x01 || UINT64_BE(L) || UINT32_BE(c).
4: Return F <- H || P.
```

**Algorithm 2B — Validation and recovery.**

```text
Require: candidate byte frame F
Ensure: original payload P or deterministic rejection
1: Reject unless F is bytes and |F| >= 17.
2: Parse (magic, version, L, expected_crc) from F[0:17].
3: Reject if magic != ICHS or version != 1.
4: Reject if |F| != 17 + L.
5: P <- F[17:17 + L].
6: Reject if CRC32(P) != expected_crc.
7: Return P.
```

**Proposition 5 — Exact frame size.** Every frame returned by Algorithm 2A has $L+17$ bytes. **Proof.** The fixed fields contribute 4, 1, 8, and 4 bytes, and concatenation appends the $L$-byte payload. Their sum is $4+1+8+4+L=L+17$. $\square$

**Proposition 6 — Round-trip correctness.** For every accepted payload, $\operatorname{unframe}(F(P))=P$. **Proof.** Algorithm 2A writes the supported constants, the exact length, $C(P)$, and then $P$. Algorithm 2B reads the same fixed offsets. Proposition 5 satisfies its length predicate, slicing selects exactly the appended $P$, and deterministic CRC recomputation matches $C(P)$; hence the returned bytes equal the input. $\square$

**Proposition 7 — Unique boundary and injectivity.** From a valid frame start and complete header, the length field identifies the unique accepted endpoint $17+L$. If $P_a\neq P_b$, then $F(P_a)\neq F(P_b)$: different lengths change the length field, while equal-length unequal payloads change the appended body. $\square$

**Proposition 8 — Bounded malformed-input rejection.** A truncated header, wrong magic, unsupported version, inconsistent length, or observed CRC mismatch activates an explicit rejection branch before return. This proposition is deliberately bounded: CRC32 collisions exist, and an attacker can alter a body and recompute its checksum. $\square$

**Capacity and overhead.** A later embedding stage must satisfy

$$C_{\mathrm{embed}}\geq8(L+17)=8L+136\text{ bits}.\tag{7}$$

For $L>0$, payload-relative overhead is $17/L$ and the header share of the complete frame is $17/(L+17)$. For $L=32$, the frame has 49 bytes or 392 bits; overhead is $53.125\%$ relative to the payload and the header share is approximately $34.694\%$. For $L=0$, a valid frame still occupies 17 bytes and payload-relative overhead is undefined.

**Complexity.** Fixed-field packing and parsing use $O(1)$ time. CRC32 and copying require $\Theta(L)$ time. Excluding returned bytes, auxiliary state is $O(1)$; the frame or recovered payload occupies $\Theta(L)$ output space.

> **Security boundary:** CRC32 provides accidental-error checking, not authenticity, secrecy, or adversarial integrity. AES-GCM and key management belong to a later I4 revision.

**Dataset boundary.** No image dataset is required for these byte-protocol proofs. Later experiments should cover empty, short UTF-8, pseudonymized structured, random, and near-capacity payloads without inserting real patient identifiers.

In [ ]:
ICHAN_MAGIC = b"ICHS"
ICHAN_VERSION = 1
ICHAN_HEADER = struct.Struct(">4sBQI")
ICHAN_MAX_PAYLOAD_BYTES = (1 << 64) - 1


def frame_size_bytes(payload_length: int) -> int:
    if isinstance(payload_length, bool) or not isinstance(payload_length, int):
        raise TypeError("payload_length must be an integer")
    if payload_length < 0 or payload_length > ICHAN_MAX_PAYLOAD_BYTES:
        raise ValueError("payload_length must fit the unsigned 64-bit field")
    return ICHAN_HEADER.size + payload_length


def required_frame_bits(payload_length: int) -> int:
    return 8 * frame_size_bytes(payload_length)


@dataclass(frozen=True)
class ICHANFrameHeader:
    magic: bytes
    version: int
    payload_length: int
    payload_crc32: int


def inspect_frame_header(frame: bytes) -> ICHANFrameHeader:
    if not isinstance(frame, bytes):
        raise TypeError("frame must be bytes")
    if len(frame) < ICHAN_HEADER.size:
        raise ValueError("frame is shorter than the 17-byte ICHAN-Secure header")
    magic, version, payload_length, payload_crc32 = ICHAN_HEADER.unpack_from(frame)
    return ICHANFrameHeader(
        magic=magic,
        version=int(version),
        payload_length=int(payload_length),
        payload_crc32=int(payload_crc32),
    )


def frame_payload(payload: bytes) -> bytes:
    if not isinstance(payload, bytes):
        raise TypeError("payload must be bytes")
    frame_size_bytes(len(payload))
    checksum = zlib.crc32(payload) & 0xFFFFFFFF
    header = ICHAN_HEADER.pack(ICHAN_MAGIC, ICHAN_VERSION, len(payload), checksum)
    return header + payload


def unframe_payload(frame: bytes) -> bytes:
    header = inspect_frame_header(frame)
    if header.magic != ICHAN_MAGIC:
        raise ValueError("invalid ICHAN-Secure magic bytes")
    if header.version != ICHAN_VERSION:
        raise ValueError(f"unsupported ICHAN-Secure version: {header.version}")
    expected_frame_length = frame_size_bytes(header.payload_length)
    if len(frame) != expected_frame_length:
        raise ValueError(
            f"frame length mismatch: declared {header.payload_length} payload bytes, "
            f"received {len(frame) - ICHAN_HEADER.size}"
        )
    payload = frame[ICHAN_HEADER.size:]
    observed_crc = zlib.crc32(payload) & 0xFFFFFFFF
    if observed_crc != header.payload_crc32:
        raise ValueError("CRC32 mismatch: payload is corrupted")
    return payload

**Worked derivation and proof mapping.** For a 32-byte payload, (6) gives $32+17=49$ frame bytes and $8(49)=392$ required embedding bits. The payload-relative overhead is $17/32=0.53125$, and the header share is $17/49\approx0.34694$.

The next unexecuted assertions map the implementation to the formal obligations:

| Assertion | Mathematical obligation |
|---|---|
| `len(frame) == len(payload) + 17` | Proposition 5 |
| recovered bytes equal source bytes | Proposition 6 |
| different payloads produce different complete frames | Proposition 7 |
| malformed candidates raise before return | Proposition 8 |
| `required_frame_bits(L) == 8*(L+17)` | capacity equation (7) |

The displayed CRC is a protocol value, not an authentication result or security score.

In [ ]:
step2_payload = b"subject=DEMO-001|study=LOGIC-CHECK|note=no-real-patient-data"
step2_frame = frame_payload(step2_payload)
step2_header = inspect_frame_header(step2_frame)
step2_recovered = unframe_payload(step2_frame)
step2_empty_frame = frame_payload(b"")
step2_distinct_frame = frame_payload(step2_payload + b"!")

assert ICHAN_HEADER.size == 17
assert len(step2_frame) == frame_size_bytes(len(step2_payload))
assert required_frame_bits(len(step2_payload)) == 8 * (len(step2_payload) + 17)
assert step2_header.magic == ICHAN_MAGIC
assert step2_header.version == ICHAN_VERSION
assert step2_header.payload_length == len(step2_payload)
assert step2_recovered == step2_payload
assert len(step2_empty_frame) == 17
assert unframe_payload(step2_empty_frame) == b""
assert step2_distinct_frame != step2_frame

step2_frame_summary = {
    "header_bytes": ICHAN_HEADER.size,
    "payload_bytes": len(step2_payload),
    "frame_bytes": len(step2_frame),
    "required_embedding_bits": required_frame_bits(len(step2_payload)),
    "payload_relative_overhead": ICHAN_HEADER.size / len(step2_payload),
    "header_share": ICHAN_HEADER.size / len(step2_frame),
    "header": step2_header,
    "exact_recovery": step2_recovered == step2_payload,
}
step2_frame_summary

**Failure-path check.** A parser contract is incomplete if only the successful path is shown. The next unexecuted cell constructs four controlled malformed frames and records their expected exceptions: truncated header, incorrect magic, unsupported version, and corrupted body. These are protocol checks, not adversarial-security tests.

In [ ]:
def capture_unframe_failure(candidate: bytes) -> dict[str, str]:
    try:
        unframe_payload(candidate)
    except (TypeError, ValueError) as error:
        return {"error_type": type(error).__name__, "message": str(error)}
    raise AssertionError("malformed frame was accepted")


step2_wrong_magic = b"FAIL" + step2_frame[4:]
step2_wrong_version = step2_frame[:4] + bytes([ICHAN_VERSION + 1]) + step2_frame[5:]
step2_corrupted_body = step2_frame[:-1] + bytes([step2_frame[-1] ^ 0b00000001])

step2_failure_summary = {
    "truncated_header": capture_unframe_failure(step2_frame[: ICHAN_HEADER.size - 1]),
    "wrong_magic": capture_unframe_failure(step2_wrong_magic),
    "unsupported_version": capture_unframe_failure(step2_wrong_version),
    "corrupted_body": capture_unframe_failure(step2_corrupted_body),
}
step2_failure_summary

**Mathematical visualization — proposed Fig. 2.** The byte-layout diagram below draws every field using its actual width for a selected payload length. Field names, offsets, sizes, borders, and hatch patterns duplicate the color encoding. The 32-byte example has total width 49 bytes, directly visualizing Proposition 5 and the endpoint used in Proposition 7. Proposed caption: **Byte-level ICHAN-Secure frame layout showing fixed header fields, offsets, and the variable-length payload.**

In [ ]:
def plot_step2_frame_layout(
    payload_length: int = 32,
) -> tuple[plt.Figure, plt.Axes]:
    total_bytes = frame_size_bytes(payload_length)
    fields = [
        ("Magic", 4, "#4c78a8", "///"),
        ("Version", 1, "#72b7b2", "..."),
        ("Length", 8, "#f2cf5b", "---"),
        ("CRC32", 4, "#f58518", "xxx"),
        ("Payload", payload_length, "#e45756", "+++"),
    ]

    figure, axis = plt.subplots(figsize=(15, 3.2), constrained_layout=True)
    offset = 0
    boundaries = [0]
    for name, width, color, hatch in fields:
        axis.barh(
            y=0, width=width, left=offset, height=0.55,
            color=color, edgecolor="black", hatch=hatch, alpha=0.75,
        )
        if width > 0:
            axis.text(
                offset + width / 2, 0, f"{name}\n{width} B",
                ha="center", va="center", fontsize=8, fontweight="bold",
            )
        offset += width
        boundaries.append(offset)

    axis.set(
        xlim=(0, max(total_bytes, 1)), ylim=(-0.6, 0.6), yticks=[],
        xlabel="byte offset from frame start",
        title=f"Step 2 frame layout: 17-byte header + {payload_length}-byte payload = {total_bytes} bytes",
    )
    axis.set_xticks(sorted(set(boundaries)))
    axis.grid(axis="x", linestyle=":", alpha=0.35)
    return figure, axis


step2_layout_figure, step2_layout_axis = plot_step2_frame_layout(32)

**Visualization choice.** Framing does not depend on image modality, so an image plot would add no evidence. The useful visualization is protocol overhead: total framed bytes and the percentage occupied by the fixed header as source-payload size changes. This plot applies unchanged to every image dataset.

In [ ]:
def plot_frame_overhead(
    payload_sizes: Iterable[int] = (0, 1, 8, 16, 32, 64, 128, 256, 512, 1024),
) -> tuple[plt.Figure, np.ndarray]:
    sizes = np.asarray(list(payload_sizes), dtype=np.int64)
    if sizes.ndim != 1 or sizes.size == 0:
        raise ValueError("payload_sizes must be a non-empty one-dimensional sequence")
    if np.any(sizes < 0):
        raise ValueError("payload sizes must not be negative")
    frame_sizes = sizes + ICHAN_HEADER.size
    header_share = 100.0 * ICHAN_HEADER.size / frame_sizes

    figure, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].plot(sizes, frame_sizes, marker="o", label="payload + 17-byte header")
    axes[0].plot(sizes, sizes, linestyle="--", color="gray", label="payload only")
    axes[0].set(
        title="Frame size", xlabel="source payload (bytes)", ylabel="framed size (bytes)"
    )
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    axes[1].plot(sizes, header_share, marker="o", color="#d95f02")
    axes[1].set(
        title="Header share of frame",
        xlabel="source payload (bytes)",
        ylabel="header share (%)",
        ylim=(0, 105),
    )
    axes[1].grid(alpha=0.25)
    return figure, axes

In [ ]:
# This protocol-overhead plot is dataset-independent.
step2_overhead_figure, step2_overhead_axes = plot_frame_overhead()

**Integration boundary after Step 2.**

The existing class below connects the completed Step 1 classifier and Step 2 frame to adaptive embedding. Its depth-map alignment, extraction traversal, and reversibility limitations belong to Step 3 and will be reviewed separately.

In [ ]:
class ICHANSecure:
    """Logic-corrected adaptive LSB baseline with framed payloads."""

    def __init__(self, *, bits_stored: int, signed: bool) -> None:
        integer_domain(bits_stored, signed)
        self.bits_stored = bits_stored
        self.signed = signed

    def labels(self, image: np.ndarray) -> tuple[np.ndarray, tuple[int, int]]:
        return classify_native_intensity(
            image, bits_stored=self.bits_stored, signed=self.signed
        )

    def embed(self, cover: np.ndarray, payload: bytes) -> EmbeddingResult:
        image = validate_grayscale_integer(cover, name="cover")
        labels, cutoffs = self.labels(image)
        counts = np.bincount(labels.ravel(), minlength=3)
        p0, p1, p2 = (int(counts[0]), int(counts[1]), int(counts[2]))
        framed_payload = frame_payload(payload)
        framed_bits = bytes_to_bits(framed_payload)
        case, capacity = choose_embedding_case(len(framed_bits), p0, p1, p2)

        original_flat = image.ravel()
        stego_flat = original_flat.copy()
        depth_flat = np.zeros(stego_flat.size, dtype=np.uint8)
        bit_index = 0

        for pixel_index, label in enumerate(labels.ravel()):
            if bit_index >= len(framed_bits):
                break
            nominal_depth = depth_for_label(case, int(label))
            if nominal_depth == 0:
                continue
            actual_depth = min(nominal_depth, len(framed_bits) - bit_index)
            chunk = framed_bits[bit_index : bit_index + actual_depth]
            new_value = replace_lsb(int(stego_flat[pixel_index]), chunk)
            stego_flat[pixel_index] = new_value
            depth_flat[pixel_index] = actual_depth
            bit_index += actual_depth

        if bit_index != len(framed_bits):
            raise RuntimeError("internal embedding invariant failed: frame is incomplete")

        stego = stego_flat.reshape(image.shape)
        return EmbeddingResult(
            stego=stego,
            depth_map=depth_flat.reshape(image.shape),
            case=case,
            payload_bits=len(framed_bits),
            capacity_bits=capacity,
            modified_pixels=int(np.count_nonzero(stego_flat != original_flat)),
            metadata={
                "source_payload_bytes": len(payload),
                "framed_payload_bytes": len(framed_payload),
                "header_bytes": ICHAN_HEADER.size,
                "class_counts": (p0, p1, p2),
                "native_cutoffs": cutoffs,
                "bits_stored": self.bits_stored,
                "signed": self.signed,
                "workflow": "ichan-secure-logic-first",
            },
        )

    @staticmethod
    def extract(stego: np.ndarray, depth_map: np.ndarray) -> bytes:
        image = validate_grayscale_integer(stego, name="stego")
        depths = np.asarray(depth_map)
        if depths.shape != image.shape:
            raise ValueError("depth_map shape must match stego image shape")
        if np.any(depths < 0) or np.any(depths > 3):
            raise ValueError("depth_map entries must be from zero through three")

        recovered_bits: list[int] = []
        for pixel_value, depth in zip(image.ravel(), depths.ravel()):
            if int(depth) > 0:
                recovered_bits.extend(extract_lsb(int(pixel_value), int(depth)))
        frame = bits_to_bytes(recovered_bits)
        return unframe_payload(frame)

**Implementation example and transition to real data.**

Use a synthetic 12-bit unsigned array only to debug domain scaling. Replace it with real DICOM data after the DICOM loader cell works. Search for **`TCIA LIDC-IDRI DICOM download`** for the primary development dataset. Record subject, study, series, SOP Instance UID, modality, dimensions, `BitsAllocated`, `BitsStored`, `PixelRepresentation`, photometric interpretation, and transfer syntax in a manifest.

In [ ]:
ichan_rng = np.random.default_rng(1163)
ichan_cover_12bit = ichan_rng.integers(0, 4096, size=(512, 512), dtype=np.uint16)
ichan_payload = (
    b"ICHAN-Secure logic-first payload | synthetic data only | "
    b"CRC32 protects accidental corruption, not malicious modification"
)

ichan_method = ICHANSecure(bits_stored=12, signed=False)
ichan_result = ichan_method.embed(ichan_cover_12bit, ichan_payload)
ichan_recovered = ichan_method.extract(ichan_result.stego, ichan_result.depth_map)
ichan_metrics = evaluate_embedding(
    ichan_cover_12bit, ichan_result, bits_stored=12, signed=False
)

ichan_example_summary = {
    "exact_recovery": ichan_recovered == ichan_payload,
    "bit_error_rate": bit_error_rate(ichan_payload, ichan_recovered),
    "case": ichan_result.case,
    "metrics": ichan_metrics,
    **ichan_result.metadata,
}
ichan_example_summary

**Embedded-frame corruption demonstration.**

Flipping one used LSB should cause header rejection or CRC32 mismatch. This demonstrates accidental-corruption detection only. An attacker can recompute CRC32; authenticated encryption is required for malicious tamper resistance.

In [ ]:
corrupted_stego = ichan_result.stego.copy()
used_positions = np.flatnonzero(ichan_result.depth_map.ravel() > 0)
if used_positions.size == 0:
    raise RuntimeError("example produced no embedded positions")
corrupted_stego.ravel()[used_positions[0]] ^= 1

try:
    ICHANSecure.extract(corrupted_stego, ichan_result.depth_map)
except ValueError as error:
    corruption_result = {"detected": True, "reason": str(error)}
else:
    corruption_result = {"detected": False, "reason": None}
corruption_result

### F. Dataset Selection and Provenance

Use sources in stages:

| Stage | Source | Search phrase | Role |
|---|---|---|---|
| Loader check | pydicom example datasets | `pydicom examples get_path ct` | Metadata and pixel-array debugging only |
| Development | TCIA LIDC-IDRI, DOI `10.7937/K9/TCIA.2015.LO9QL9SX` | `TCIA LIDC-IDRI DICOM download` | Patient-level CT development sample |
| External validation | TCIA TCGA-LUAD, DOI `10.7937/K9/TCIA.2016.JGNIHEP5` | `TCIA TCGA-LUAD DICOM` | Unseen heterogeneous CT subset |
| Optional modality shift | MIMIC-CXR DICOM, DOI `10.13026/C2JT1Q` | `PhysioNet MIMIC-CXR credentialed access` | Chest radiograph validation under its data-use agreement |

Recommended starting protocol: at least 30 development patients and 30 different external-validation patients. This is a practical starting recommendation, not a universal power calculation. Keep slices grouped by patient and report patient-level summaries.

#### DICOM handling logic

The loader accepts only single-frame monochrome integer images in this revision. The saver accepts only uncompressed little-endian inputs, refuses existing digital signatures, creates a new SOP Instance UID, and marks the result as derived. These restrictions avoid silently producing a misleading file. Multiframe, color, compressed, big-endian, floating-point, and signedness-conversion workflows are future work.

Embedding operates on stored pixel values. Modality rescale and VOI windowing are for interpretation and display; they must not silently replace the raw array used for reversible or bit-level reasoning.

In [ ]:
@dataclass(frozen=True)
class DicomImage:
    dataset: Any
    pixels: np.ndarray
    bits_stored: int
    signed: bool
    source_path: Path


def require_pydicom() -> None:
    if pydicom is None:
        raise ImportError("install pydicom>=3.0 before using DICOM cells")


def load_dicom_image(path: str | Path) -> DicomImage:
    require_pydicom()
    source_path = Path(path)
    dataset = pydicom.dcmread(source_path)
    if int(getattr(dataset, "SamplesPerPixel", 1)) != 1:
        raise ValueError("this notebook revision supports only monochrome DICOM images")
    pixels = np.asarray(dataset.pixel_array)
    if pixels.ndim != 2:
        raise ValueError("this notebook revision supports only single-frame DICOM images")
    validate_grayscale_integer(pixels, name="DICOM pixel array")
    bits_stored = int(dataset.BitsStored)
    signed = int(dataset.PixelRepresentation) == 1
    integer_domain(bits_stored, signed)
    return DicomImage(
        dataset=dataset,
        pixels=pixels,
        bits_stored=bits_stored,
        signed=signed,
        source_path=source_path,
    )


def dicom_manifest_row(image: DicomImage) -> dict[str, Any]:
    dataset = image.dataset
    transfer_syntax = getattr(getattr(dataset, "file_meta", None), "TransferSyntaxUID", None)
    return {
        "source_path": str(image.source_path),
        "study_uid": str(getattr(dataset, "StudyInstanceUID", "")),
        "series_uid": str(getattr(dataset, "SeriesInstanceUID", "")),
        "sop_instance_uid": str(getattr(dataset, "SOPInstanceUID", "")),
        "modality": str(getattr(dataset, "Modality", "")),
        "rows": int(getattr(dataset, "Rows", image.pixels.shape[0])),
        "columns": int(getattr(dataset, "Columns", image.pixels.shape[1])),
        "bits_allocated": int(getattr(dataset, "BitsAllocated", image.pixels.dtype.itemsize * 8)),
        "bits_stored": image.bits_stored,
        "pixel_representation": int(image.signed),
        "photometric_interpretation": str(getattr(dataset, "PhotometricInterpretation", "")),
        "transfer_syntax_uid": str(transfer_syntax or ""),
    }


def save_derived_dicom(
    source: DicomImage, stego: np.ndarray, output_path: str | Path
) -> Path:
    require_pydicom()
    modified = validate_grayscale_integer(stego, name="stego")
    if modified.shape != source.pixels.shape:
        raise ValueError("stego shape must match source DICOM pixel array")
    if modified.dtype != source.pixels.dtype:
        raise TypeError("stego dtype must match source DICOM pixel dtype")

    dataset = copy.deepcopy(source.dataset)
    transfer_syntax = dataset.file_meta.TransferSyntaxUID
    if transfer_syntax.is_compressed:
        raise ValueError("compressed transfer syntaxes require an explicit re-encoding workflow")
    if dataset.is_little_endian is False:
        raise ValueError("big-endian DICOM output is outside this notebook's scope")
    if "DigitalSignaturesSequence" in dataset:
        raise ValueError("pixel modification would invalidate the existing DICOM digital signature")

    raw_pixels = np.ascontiguousarray(modified).tobytes()
    if len(raw_pixels) % 2 == 1:
        raw_pixels += b"\x00"
    dataset.PixelData = raw_pixels
    dataset.Rows, dataset.Columns = modified.shape
    dataset.SOPInstanceUID = generate_uid()
    dataset.file_meta.MediaStorageSOPInstanceUID = dataset.SOPInstanceUID
    dataset.ImageType = ["DERIVED", "SECONDARY"]
    dataset.SeriesDescription = "ICHAN-Secure research-derived image"
    dataset.ImageComments = "Research output; not validated for diagnostic use"

    destination = Path(output_path)
    dataset.save_as(destination, enforce_file_format=True)
    return destination

Replace the placeholder path below with one local DICOM file. Inspect `dicom_info` before embedding. If `BitsStored`, signedness, modality, or transfer syntax is unexpected, stop and document it rather than converting silently.

In [ ]:
DICOM_PATH = Path("data/replace-with-a-single-uncompressed-monochrome-image.dcm")

if not DICOM_PATH.exists():
    raise FileNotFoundError(
        f"Set DICOM_PATH to a real file before running this cell: {DICOM_PATH}"
    )

dicom_image = load_dicom_image(DICOM_PATH)
dicom_info = dicom_manifest_row(dicom_image)
dicom_info

In [ ]:
dicom_method = ICHANSecure(
    bits_stored=dicom_image.bits_stored,
    signed=dicom_image.signed,
)
dicom_payload = b"Synthetic research payload; no protected health information"
dicom_result = dicom_method.embed(dicom_image.pixels, dicom_payload)
dicom_recovered = dicom_method.extract(dicom_result.stego, dicom_result.depth_map)
dicom_metrics = evaluate_embedding(
    dicom_image.pixels,
    dicom_result,
    bits_stored=dicom_image.bits_stored,
    signed=dicom_image.signed,
)

dicom_run_summary = {
    **dicom_info,
    "exact_recovery": dicom_recovered == dicom_payload,
    "bit_error_rate": bit_error_rate(dicom_payload, dicom_recovered),
    "case": dicom_result.case,
    "metrics": dicom_metrics,
}
dicom_run_summary

### G. Controlled Comparison and Payload Sweep

For a fair comparison, both workflows must receive the same 8-bit cover and same payload bytes. Native 12- or 16-bit experiments evaluate ICHAN-Secure separately because Workflow 1 is explicitly 8-bit. Payload curves should use bits per pixel on the x-axis. Generate independent random payload bytes with a recorded seed; do not repeat Lorem Ipsum and treat it as independent trials.

In [ ]:
def compare_workflows_uint8(cover: np.ndarray, payload: bytes) -> list[dict[str, Any]]:
    image = validate_grayscale_integer(cover, name="cover")
    if image.dtype != np.uint8:
        raise TypeError("controlled workflow comparison requires uint8 cover data")

    paper = PaperFaithfulBASMEDSecure()
    paper_output = paper.embed(image, payload)
    paper_recovered = paper.extract(
        paper_output.stego, paper_output.depth_map, paper_output.payload_bits
    )
    paper_evaluation = evaluate_embedding(
        image, paper_output, bits_stored=8, signed=False
    )

    ichan = ICHANSecure(bits_stored=8, signed=False)
    ichan_output = ichan.embed(image, payload)
    ichan_recovered = ichan.extract(ichan_output.stego, ichan_output.depth_map)
    ichan_evaluation = evaluate_embedding(
        image, ichan_output, bits_stored=8, signed=False
    )

    return [
        {
            "method": "BASMEDSecure reproduction",
            "case": paper_output.case,
            "exact_recovery": paper_recovered == payload,
            "bit_error_rate": bit_error_rate(payload, paper_recovered),
            **paper_evaluation.__dict__,
        },
        {
            "method": "ICHAN-Secure logic-first",
            "case": ichan_output.case,
            "exact_recovery": ichan_recovered == payload,
            "bit_error_rate": bit_error_rate(payload, ichan_recovered),
            **ichan_evaluation.__dict__,
        },
    ]


def run_ichan_payload_sweep(
    cover: np.ndarray,
    *,
    bits_stored: int,
    signed: bool,
    payload_sizes_bytes: Iterable[int],
    seed: int = 20260901,
) -> list[dict[str, Any]]:
    image = validate_grayscale_integer(cover, name="cover")
    rng = np.random.default_rng(seed)
    method = ICHANSecure(bits_stored=bits_stored, signed=signed)
    records: list[dict[str, Any]] = []

    for payload_bytes in payload_sizes_bytes:
        if payload_bytes < 0:
            raise ValueError("payload sizes must not be negative")
        payload = rng.integers(0, 256, size=payload_bytes, dtype=np.uint8).tobytes()
        result = method.embed(image, payload)
        recovered = method.extract(result.stego, result.depth_map)
        metrics = evaluate_embedding(
            image, result, bits_stored=bits_stored, signed=signed
        )
        records.append({
            "payload_source_bytes": payload_bytes,
            "payload_framed_bits": result.payload_bits,
            "case": result.case,
            "exact_recovery": recovered == payload,
            "bit_error_rate": bit_error_rate(payload, recovered),
            **metrics.__dict__,
        })
    return records

In [ ]:
def plot_payload_quality(records: list[dict[str, Any]]) -> tuple[plt.Figure, np.ndarray]:
    if not records:
        raise ValueError("records must not be empty")
    ordered = sorted(records, key=lambda row: row["payload_bpp"])
    bpp = [row["payload_bpp"] for row in ordered]
    psnr_values = [row["psnr_db"] for row in ordered]
    ssim_values = [row["ssim"] for row in ordered]
    overhead_values = [row["depth_map_overhead_ratio"] for row in ordered]

    figure, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    axes[0].plot(bpp, psnr_values, marker="o")
    axes[0].set_ylabel("PSNR (dB)")
    axes[1].plot(bpp, ssim_values, marker="o")
    axes[1].set_ylabel("SSIM")
    axes[2].plot(bpp, overhead_values, marker="o")
    axes[2].set_ylabel("Packed depth-map bits / payload bits")
    for axis in axes:
        axis.set_xlabel("Payload (bits per pixel)")
        axis.grid(alpha=0.25)
    figure.suptitle("Payload, image quality, and side-information cost")
    return figure, axes

#### Dataset-dependent batch workflow

Build a manifest before experiments. Sample patients first, then slices within each selected patient. Do not randomly split slices across development and validation because slices from one patient are correlated. The scaffold below discovers `.dcm` files only; collections without extensions need a manifest supplied by their download tool.

In [ ]:
def discover_dicom_files(root: str | Path) -> list[Path]:
    data_root = Path(root).resolve()
    if not data_root.exists() or not data_root.is_dir():
        raise FileNotFoundError(f"DICOM root does not exist: {data_root}")
    return sorted(path for path in data_root.rglob("*.dcm") if path.is_file())


def run_ichan_dicom_record(path: str | Path, payload: bytes) -> dict[str, Any]:
    source = load_dicom_image(path)
    method = ICHANSecure(bits_stored=source.bits_stored, signed=source.signed)
    result = method.embed(source.pixels, payload)
    recovered = method.extract(result.stego, result.depth_map)
    metrics = evaluate_embedding(
        source.pixels,
        result,
        bits_stored=source.bits_stored,
        signed=source.signed,
    )
    return {
        **dicom_manifest_row(source),
        "payload_source_bytes": len(payload),
        "payload_framed_bits": result.payload_bits,
        "case": result.case,
        "exact_recovery": recovered == payload,
        "bit_error_rate": bit_error_rate(payload, recovered),
        **metrics.__dict__,
    }


DATASET_ROOT = Path("data/replace-with-downloaded-dicom-root")
EXPERIMENT_PAYLOAD = b"Synthetic experiment payload; never use real patient information"

# Run these lines only after DATASET_ROOT and a patient-level selection manifest are ready.
# selected_paths = discover_dicom_files(DATASET_ROOT)
# experiment_records = [run_ichan_dicom_record(path, EXPERIMENT_PAYLOAD) for path in selected_paths]


## IV. Results and Discussion

No result is inserted until debugging and experiments are complete. Use these subsections in the paper.

### A. Reproduction Results

Report exact recovery, classifier-threshold agreement, absent-class failures, and every interpretation required to execute the paper.

### B. Correctness and Extraction Results

Report framed recovery, CRC behavior, malformed inputs, oversized payloads, empty payloads, and final partial chunks.

### C. Capacity and Image-Quality Results

Report bpp, capacity utilization, map overhead, modified-pixel fraction, maximum change, MSE, PSNR, and SSIM by patient, dataset, bit depth, payload, and case.

### D. Security Evaluation Scope and Outstanding Tests

Until steganalysis is implemented, use this cautious subsection title. Do not interpret PSNR or CRC32 as confidentiality, authentication, or resistance to detection.

### E. DICOM and Clinical-Preservation Results

Report metadata preservation, transfer-syntax restrictions, signedness, bit depth, raw-value changes, ROI metrics, and digital-signature implications. Clinical claims require task-based or expert evaluation.

### F. Threats to Validity

Address patient correlation, dataset size, modality imbalance, source mismatch, payload generation, metric limitations, code interpretation, side-map transfer, and missing clinical evidence.

### Results table template

Create one row per image and payload, then aggregate by patient before computing study-level uncertainty. Suggested columns:

`dataset`, `subject`, `study_uid`, `series_uid`, `sop_instance_uid`, `method`, `bits_stored`, `signed`, `payload_source_bytes`, `payload_bpp`, `case`, `capacity_bits`, `exact_recovery`, `bit_error_rate`, `crc_valid`, `modified_fraction`, `max_absolute_change`, `mse`, `psnr_db`, `ssim`, `depth_map_overhead_ratio`, `runtime_ms`, `notes`.

Do not publish private patient tags. Follow dataset licenses for permitted identifiers.

## V. Conclusion and Future Work

This logic-first revision separates reproduction from correction. Workflow 1 preserves the paper's defining logistic-regression and adaptive-LSB behavior while disclosing necessary pseudocode repairs. Workflow 2 removes redundant model training, supports native integer domains, embeds payload length, and validates accidental corruption. Neither workflow is yet reversible, cryptographically authenticated, statistically evaluated against steganalysis, or clinically validated.

Recommended next order after debugging:

1. run matched patient-level experiments;
2. add keyed placement and AES-GCM;
3. replace irreversible LSB replacement with reversible prediction-error or histogram-shifting logic;
4. add RS/SRM or learned steganalysis;
5. add ROI/task-based clinical evaluation;
6. add ML-KEM key wrapping;
7. build a small NEQR quantum-circuit proof of concept and count preparation, gates, noise, and readout before claiming acceleration.

## Acknowledgment

Add confirmed funding, laboratory support, dataset acknowledgments, and clinical collaborators. Follow TCIA and PhysioNet acknowledgment requirements exactly. Do not list contributors without their approval.

## References

[1] B. A. Salim *et al.*, “BASMEDSecure: A Logistic Regression-Based Data Hiding Framework for Securing Patient Records,” local manuscript, 2025.

[2] J. Fridrich, M. Goljan, and R. Du, “Reliable detection of LSB steganography in color and grayscale images,” 2001, doi: [10.1145/1232454.1232466](https://doi.org/10.1145/1232454.1232466).

[3] V. Holub, J. Fridrich, and T. Denemark, “Universal distortion function for steganography in an arbitrary domain,” 2014, doi: [10.1186/1687-417X-2014-1](https://doi.org/10.1186/1687-417X-2014-1).

[4] J. Tian, “Reversible data embedding using a difference expansion,” 2003, doi: [10.1109/TCSVT.2003.815962](https://doi.org/10.1109/TCSVT.2003.815962).

[5] Z. Ni *et al.*, “Reversible data hiding,” 2006, doi: [10.1109/TCSVT.2006.869964](https://doi.org/10.1109/TCSVT.2006.869964).

[6] M. A. Ahmad *et al.*, “Hiding patients' medical reports using an enhanced wavelet steganography algorithm in DICOM images,” 2022, doi: [10.1016/j.aej.2022.03.056](https://doi.org/10.1016/j.aej.2022.03.056).

[7] H. Zhang, S. Sun, and F. Meng, “A high-capacity and reversible patient data hiding scheme for telemedicine,” 2022, doi: [10.1016/j.bspc.2022.103706](https://doi.org/10.1016/j.bspc.2022.103706).

[8] A. Mason *et al.*, “Comparison of objective image quality metrics to expert radiologists' scoring of diagnostic quality of MR images,” 2020, doi: [10.1109/TMI.2019.2930338](https://doi.org/10.1109/TMI.2019.2930338).

[9] DICOM Standards Committee, “DICOM PS3.5: Data Structures and Encoding,” current edition: [DICOM PS3.5](https://dicom.nema.org/medical/dicom/current/output/pdf/part05.pdf).

[10] NIST, “Module-Lattice-Based Key-Encapsulation Mechanism Standard,” FIPS 203, 2024, doi: [10.6028/NIST.FIPS.203](https://doi.org/10.6028/NIST.FIPS.203).

[11] Y. Zhang *et al.*, “NEQR: A novel enhanced quantum representation of digital images,” 2013, doi: [10.1007/s11128-013-0567-z](https://doi.org/10.1007/s11128-013-0567-z).

[12] N. Jiang, N. Zhao, and L. Wang, “LSB based quantum image steganography algorithm,” 2016, doi: [10.1007/s10773-015-2640-0](https://doi.org/10.1007/s10773-015-2640-0).

[13] S. G. Armato III *et al.*, “Data From LIDC-IDRI,” TCIA, 2015, doi: [10.7937/K9/TCIA.2015.LO9QL9SX](https://doi.org/10.7937/K9/TCIA.2015.LO9QL9SX).

[14] B. Albertina *et al.*, “The Cancer Genome Atlas Lung Adenocarcinoma Collection,” TCIA, 2016, doi: [10.7937/K9/TCIA.2016.JGNIHEP5](https://doi.org/10.7937/K9/TCIA.2016.JGNIHEP5).

[15] A. Johnson *et al.*, “MIMIC-CXR Database,” PhysioNet, version 2.0.0, 2019, doi: [10.13026/C2JT1Q](https://doi.org/10.13026/C2JT1Q).